# BPE 实验（2.6）

本 Notebook 只实现作业 2.6 的必需内容：
- 在 TinyStories 上训练 10K BPE，并记录时间/内存/最长词元
- 在 OpenWebText 上训练 32K BPE，并记录最长词元
- 序列化 `vocab` 与 `merges` 到磁盘
- 给出 2.6(a)(b) 对应的 1-2 句话交付内容

不添加与作业无关的功能。

In [7]:
from __future__ import annotations
import io
import os
import cProfile
import pstats
import importlib
from pathlib import Path

from tqdm import tqdm

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'cs336_basics').exists():
    ROOT = ROOT.parent
if str(ROOT) not in os.sys.path:
    os.sys.path.insert(0, str(ROOT))

train_bpe_mod = importlib.import_module('cs336_basics.tokenizer.train_bpe')
train_bpe_mod = importlib.reload(train_bpe_mod)
train_bpe = train_bpe_mod.train_bpe
train_bpe_fast = train_bpe_mod.train_bpe_fast

utils_mod = importlib.import_module('cs336_basics.tokenizer.utils')
utils_mod = importlib.reload(utils_mod)
longest_token = utils_mod.longest_token
make_profile_subset = utils_mod.make_profile_subset
measure_peak_rss_during = utils_mod.measure_peak_rss_during
save_vocab_and_merges = utils_mod.save_vocab_and_merges


def train_bpe_notebook_progress(
    input_path,
    vocab_size,
    special_tokens=None,
    progress_desc='BPE merges',
):
    special_tokens = special_tokens or []

    if vocab_size <= 0:
        raise ValueError('vocab_size must be positive')

    vocab = {index: bytes([index]) for index in range(256)}
    existing_tokens = set(vocab.values())

    for token in special_tokens:
        token_bytes = token.encode('utf-8')
        if token_bytes not in existing_tokens:
            vocab[len(vocab)] = token_bytes
            existing_tokens.add(token_bytes)

    if len(vocab) >= vocab_size:
        return vocab, []

    print(f'[BPE] {progress_desc}: building word counter...', flush=True)
    word_counter = train_bpe_mod._build_word_counter_from_file_fast(input_path, special_tokens)
    print(f'[BPE] {progress_desc}: word counter done, unique words={len(word_counter)}', flush=True)

    print(f'[BPE] {progress_desc}: building pair stats...', flush=True)
    pair_counter, pair_to_words = train_bpe_mod._build_pair_stats(word_counter)
    print(f'[BPE] {progress_desc}: pair stats done, unique pairs={len(pair_counter)}', flush=True)

    merges = []
    pair_heap = train_bpe_mod.build_pair_heap(pair_counter, vocab)
    merges_to_learn = vocab_size - len(vocab)

    for _ in tqdm(
        range(merges_to_learn),
        total=merges_to_learn,
        desc=progress_desc,
        unit='merge',
        dynamic_ncols=True,
        mininterval=0.5,
        smoothing=0.1,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]',
    ):
        if not pair_counter:
            break

        pair = train_bpe_mod.pop_most_frequent_pair(pair_heap, pair_counter)
        token_1, token_2 = pair

        merge_bytes = (vocab[token_1], vocab[token_2])
        merges.append(merge_bytes)

        new_id = len(vocab)
        vocab[new_id] = merge_bytes[0] + merge_bytes[1]

        word_counter, pair_counter, pair_heap, pair_to_words = train_bpe_mod.merge_pairs_with_heap_index(
            word_counter=word_counter,
            pair_counter=pair_counter,
            target_pair=pair,
            new_id=new_id,
            vocab=vocab,
            pair_heap=pair_heap,
            pair_to_words=pair_to_words,
        )

    return vocab, merges


DATA_DIR = Path('/mnt/dataset0/gjt/CS336/dataset')
OUT_DIR = ROOT / 'artifacts' / 'bpe'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TINY_TRAIN = DATA_DIR / 'TinyStoriesV2-GPT4-train.txt'
OWT_TRAIN = DATA_DIR / 'owt_train.txt'
SPECIAL = ['<|endoftext|>']

print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)
print('TinyStories exists:', TINY_TRAIN.exists())
print('OWT exists:', OWT_TRAIN.exists())
print('train_bpe_fast loaded:', callable(train_bpe_fast))

ROOT = /mnt/dataset0/gjt/CS336/assignment1-basics-main
DATA_DIR = /mnt/dataset0/gjt/CS336/dataset
TinyStories exists: True
OWT exists: True
train_bpe_fast loaded: True


## 2.6(a) TinyStories：训练 10K BPE 并记录指标

说明：
1. 使用 `special_tokens=['<|endoftext|>']`。
2. 记录训练总时长、峰值 RSS 内存。
3. 保存 `vocab.json` 与 `merges.txt` 到 `artifacts/bpe/tinystories_10k/`。
4. 统计词汇表中最长词元。

In [8]:
assert TINY_TRAIN.exists(), f'Missing: {TINY_TRAIN}'
tiny_dir = OUT_DIR / 'tinystories_10k'
tiny_dir.mkdir(parents=True, exist_ok=True)

(tiny_vocab, tiny_merges), tiny_secs, tiny_peak = measure_peak_rss_during(
    lambda: train_bpe_notebook_progress(
        TINY_TRAIN,
        vocab_size=10_000,
        special_tokens=SPECIAL,
        progress_desc='TinyStories 10K merge progress',
    )
)
save_vocab_and_merges(tiny_vocab, tiny_merges, tiny_dir)
tiny_longest = longest_token(tiny_vocab)

print('tiny_secs:', round(tiny_secs, 2))
print('tiny_peak_GB:', round(tiny_peak / (1024**3), 3))
print('tiny_vocab_size:', len(tiny_vocab), 'tiny_merges:', len(tiny_merges))
print('tiny_longest_bytes:', tiny_longest[1])
print('tiny_longest_text:', repr(tiny_longest[2][:120]))

[BPE] TinyStories 10K merge progress: building word counter...
[BPE] TinyStories 10K merge progress: word counter done, unique words=59933
[BPE] TinyStories 10K merge progress: building pair stats...
[BPE] TinyStories 10K merge progress: pair stats done, unique pairs=2108


TinyStories 10K merge progress: 100%|██████████| 9743/9743 [00:36<00:00, 264.54merge/s]


[SAVE] vocab  → /mnt/dataset0/gjt/CS336/assignment1-basics-main/artifacts/bpe/tinystories_10k/vocab.json
[SAVE] merges → /mnt/dataset0/gjt/CS336/assignment1-basics-main/artifacts/bpe/tinystories_10k/merges.txt
tiny_secs: 443.74
tiny_peak_GB: 4.758
tiny_vocab_size: 10000 tiny_merges: 9743
tiny_longest_bytes: 15
tiny_longest_text: ' accomplishment'


## 2.6(b) TinyStories：代码性能分析（瓶颈）

说明：使用 `cProfile` 对训练过程做函数级耗时统计，提取累计时间最高的函数作为瓶颈候选。

In [9]:
# 2.6(b) profiling：保留进度条显示（使用子集避免耗时过长）
subset_path = OUT_DIR / 'profile_subset_64mb.txt'
_ = make_profile_subset(TINY_TRAIN, subset_path, max_bytes=64 * 1024 * 1024)

pr = cProfile.Profile()
pr.enable()
_ = train_bpe_notebook_progress(
    subset_path,
    vocab_size=2_000,
    special_tokens=SPECIAL,
    progress_desc='TinyStories profile 2K merge progress',
)
pr.disable()

s = io.StringIO()
stats = pstats.Stats(pr, stream=s).sort_stats('cumulative')
stats.print_stats(20)
print(s.getvalue())

hot = []
for (file, line, fn), info in stats.stats.items():
    cc, nc, tt, ct, callers = info
    if 'cs336_basics/tokenizer' in file.replace('\\', '/'):
        hot.append((ct, Path(file).name, fn, line))
hot.sort(reverse=True)
print('Top tokenizer hotspot:', hot[0] if hot else 'N/A')
print('Profile dataset:', subset_path)
print('Profile vocab_size:', 2000)

[BPE] TinyStories profile 2K merge progress: building word counter...
[BPE] TinyStories profile 2K merge progress: word counter done, unique words=18909
[BPE] TinyStories profile 2K merge progress: building pair stats...
[BPE] TinyStories profile 2K merge progress: pair stats done, unique pairs=1216


TinyStories profile 2K merge progress: 100%|██████████| 1743/1743 [00:03<00:00, 532.13merge/s]

         56525957 function calls (56525694 primitive calls) in 25.947 seconds

   Ordered by: cumulative time
   List reduced from 335 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    66/65    0.012    0.000   58.866    0.906 /home/zhichao/miniconda3/envs/ljz/lib/python3.12/asyncio/base_events.py:1922(_run_once)
       66    0.042    0.001   23.817    0.361 /home/zhichao/miniconda3/envs/ljz/lib/python3.12/asyncio/events.py:86(_run)
 16165714    8.040    0.000   10.849    0.000 /mnt/dataset0/gjt/CS336/assignment1-basics-main/cs336_basics/tokenizer/train_bpe.py:151(_iter_pretokens_fast)
    66/65    2.903    0.044    7.685    0.118 /home/zhichao/miniconda3/envs/ljz/lib/python3.12/selectors.py:451(select)
        4    2.852    0.713    6.897    1.724 /mnt/dataset0/gjt/CS336/assignment1-basics-main/cs336_basics/tokenizer/train_bpe.py:179(_build_word_counter_fast)
      494    0.946    0.002    5.875    0.012 {built-in method time.sl